# PH2_NB6c — Statistical-Feature Ablation over the Full Track-A Protocol

One of the five statistical features is suspected of hurting the hybrid (TTR's class direction is
reversed on this corpus, and DeepSeek's LOGO fold regressed under fusion). This notebook identifies
the culprit — or clears all five — by brute force: it re-runs the **entire Track-A experiment**
(standard split + full 6-fold LOGO) for **every feature subset** down to two active features.

## Configurations

| Disabled | Count | Active features |
|---|---|---|
| 0 (baseline) | 1 | all 5 |
| 1 each | 5 | 4 |
| every pair | 10 | 3 |
| every triple | 10 | 2 |
| all 5 (reference floor) | 1 | 0 = neural-only probe |

**27 configurations x 7 fits each (1 standard + 6 LOGO folds) = 189 trainings.**

## Measurement arm

The primary arm is the **converged linear head** (the official Track-A configuration from NB6b).
Two reasons: it is the configuration the thesis argues from, and it is deterministic — no training
stochasticity, so half-point differences between feature subsets are real, not seed noise. An
optional MLP arm (`RUN_MLP_ARM = True`, GPU recommended) repeats everything with the fixed MLP head
(256, dropout 0.0, LayerNorm) for a robustness check of the verdict.

## How the verdict is reached (three angles, pre-declared)

1. **Leave-one-out deltas** vs the all-5 baseline: if removing feature X *improves* LOGO-worst /
   standard F1, X is harmful.
2. **Aggregated marginal contribution**: for each feature, the mean of every metric over all
   subsets that *contain* it minus the mean over all subsets that *lack* it (a Shapley-flavored
   average across contexts). A consistently negative margin across contexts is stronger evidence
   than a single deletion.
3. **Best subset overall** by the pre-registered criterion (LOGO worst fold), with its standard
   score — the candidate feature set for the final model if it beats the all-5 baseline.

**Inputs:** the same three datasets (`aigt-dataset`, `aigt-vstat`, `aigt-camelbert-cls`).
**Outputs:** `ph2_nb6c_ablation.parquet` (one row per configuration, all metrics + per-fold F1),
`ph2_nb6c_marginals.parquet`.

## Setup and alignment (same contract)

In [1]:
import pandas as pd, numpy as np, os, glob, time, json
from itertools import combinations

SEED = 42
np.random.seed(SEED)
OUT_DIR = '/kaggle/working'
RUN_MLP_ARM = True        # True -> repeat everything with the fixed MLP head (GPU recommended)

def find_file(preferred, pattern, *keywords):
    if os.path.exists(preferred): return preferred
    for kw in keywords:
        hits = [p for p in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA  = find_file('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', '*.parquet', 'dataset')
VSTAT = find_file('/kaggle/input/notebooks/bahaaqassem/nb4-extract-vstat/vstat_scaled.parquet', '*.parquet', 'vstat_scaled', 'vstat')
EMB   = find_file('/kaggle/input/notebooks/bahaaqassem/nb5c-electra-camelbert/nb5c_camelbert_msa_cls_frozen.npy', '*.npy',
                  'camelbert_msa_cls', 'camelbert')

df  = pd.read_parquet(DATA)
vs  = pd.read_parquet(VSTAT)
emb = np.load(EMB).astype(np.float32)
FEATURES = ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']

assert len(df) == len(emb)
vs_aligned = vs.set_index('article_id').loc[df['article_id']].reset_index()
assert (vs_aligned['article_id'].to_numpy() == df['article_id'].to_numpy()).all()
assert (vs_aligned['label'].to_numpy() == df['label'].to_numpy()).all()
Xstat = vs_aligned[FEATURES].to_numpy(dtype=np.float32)
y = df['label'].to_numpy(); splits = df['split'].to_numpy(); gens = df['generator'].to_numpy()
tr_m, va_m, te_m = splits=='train', splits=='val', splits=='test'
gen_list = sorted(df.loc[df['label']==1, 'generator'].unique().tolist())
print('ALIGNMENT OK |', emb.shape, Xstat.shape, '| generators:', gen_list)

ALIGNMENT OK | (7101, 768) (7101, 5) | generators: ['deepseek', 'gemini', 'gpt', 'opus', 'qwen', 'sonnet']


## One full Track-A run as a function of the active feature set

`run_trackA(active)` builds the input matrix (768 neural dims + the active statistical columns),
fits the converged linear head on the standard split, then runs all six LOGO folds. Returns every
number the analysis needs. Deterministic: same seed, LBFGS to convergence, no sampling anywhere.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def build_X(active):
    if len(active) == 0:
        return emb
    return np.concatenate([emb, Xstat[:, list(active)]], axis=1)

def fit_score_lr(Xtr, ytr, Xte, yte_):
    m = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
    m.fit(Xtr, ytr)
    proba = m.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return f1_score(yte_, pred, average='macro'), roc_auc_score(yte_, proba)

def run_trackA(active):
    X = build_X(active)
    std_f1, std_auc = fit_score_lr(X[tr_m], y[tr_m], X[te_m], y[te_m])
    folds = {}
    for g in gen_list:
        trm = (splits=='train') & ((y==0) | (gens!=g))
        tem = (splits=='test')  & ((y==0) | (gens==g))
        f1g, _ = fit_score_lr(X[trm], y[trm], X[tem], y[tem])
        folds[g] = f1g
    fv = np.array(list(folds.values()))
    return {'standard_f1': std_f1, 'standard_auc': std_auc,
            'logo_mean': fv.mean(), 'logo_worst': fv.min(),
            'worst_gen': gen_list[int(fv.argmin())],
            **{f'fold_{g}': folds[g] for g in gen_list}}

print('runner ready | one config = 7 converged LR fits')

runner ready | one config = 7 converged LR fits


## Run all 27 configurations

Ordered so the important rows print first: the all-5 baseline, then leave-one-out, then pairs,
triples, and the neural-only floor. Progress prints one line per configuration with the two numbers
that matter most (standard F1, LOGO worst).

In [3]:
configs = [tuple(range(5))]                                        # all 5
configs += [tuple(sorted(set(range(5)) - {i})) for i in range(5)]     # drop 1
configs += [tuple(sorted(set(range(5)) - set(p))) for p in combinations(range(5), 2)]
configs += [tuple(sorted(set(range(5)) - set(t))) for t in combinations(range(5), 3)]
configs += [tuple()]                                                  # none (probe floor)

def label_of(active):
    if len(active) == 5: return 'ALL 5'
    if len(active) == 0: return 'NONE (probe)'
    off = [FEATURES[i] for i in range(5) if i not in active]
    return 'without: ' + ' + '.join(off)

rows, t0 = [], time.time()
for k, active in enumerate(configs):
    r = run_trackA(active)
    rows.append({'label': label_of(active),
                 'n_active': len(active),
                 'active_idx': ','.join(map(str, active)),
                 **r})
    print(f'[{k+1:2d}/27] {label_of(active):<55} std {100*r["standard_f1"]:5.1f}  '
          f'worst {100*r["logo_worst"]:5.1f} ({r["worst_gen"]})', flush=True)
print(f'\ntotal {time.time()-t0:.0f}s')

ab = pd.DataFrame(rows)
ab.to_parquet(f'{OUT_DIR}/ph2_nb6c_ablation.parquet', index=False)

[ 1/27] ALL 5                                                   std  99.6  worst  94.2 (gpt)
[ 2/27] without: targeted_ppl                                   std  99.6  worst  94.3 (gpt)
[ 3/27] without: burstiness                                     std  99.7  worst  94.2 (gpt)
[ 4/27] without: ttr                                            std  99.8  worst  93.2 (gpt)
[ 5/27] without: entity_density                                 std  99.7  worst  94.2 (gpt)
[ 6/27] without: discourse_coherence                            std  99.7  worst  93.7 (gpt)
[ 7/27] without: targeted_ppl + burstiness                      std  99.7  worst  93.7 (gpt)
[ 8/27] without: targeted_ppl + ttr                             std  99.6  worst  92.7 (gpt)
[ 9/27] without: targeted_ppl + entity_density                  std  99.6  worst  94.2 (gpt)
[10/27] without: targeted_ppl + discourse_coherence             std  99.6  worst  94.7 (gpt)
[11/27] without: burstiness + ttr                               std  9

## Analysis 1 — Leave-one-out deltas vs the all-5 baseline

Positive delta after removing a feature = that feature was hurting.

In [4]:
base = ab[ab['label']=='ALL 5'].iloc[0]
print(f"baseline (ALL 5): std {100*base['standard_f1']:.1f} | "
      f"logo mean {100*base['logo_mean']:.1f} | worst {100*base['logo_worst']:.1f} ({base['worst_gen']})\n")

loo = ab[(ab['n_active']==4)].copy()
loo['removed'] = loo['label'].str.replace('without: ', '', regex=False)
for c, bc in [('standard_f1','standard_f1'), ('logo_mean','logo_mean'), ('logo_worst','logo_worst'),
              ('fold_gpt','fold_gpt'), ('fold_deepseek','fold_deepseek')]:
    loo[f'd_{c}'] = 100*(loo[c] - base[bc])

print('leave-one-out deltas (percentage points; + means removal HELPED):')
cols = ['removed','d_standard_f1','d_logo_mean','d_logo_worst','d_fold_gpt','d_fold_deepseek']
print(loo[cols].round(2).sort_values('d_logo_worst', ascending=False).to_string(index=False))

baseline (ALL 5): std 99.6 | logo mean 98.0 | worst 94.2 (gpt)

leave-one-out deltas (percentage points; + means removal HELPED):
            removed  d_standard_f1  d_logo_mean  d_logo_worst  d_fold_gpt  d_fold_deepseek
       targeted_ppl           0.00         0.01          0.08        0.08             0.00
         burstiness           0.09         0.44          0.00        0.00             0.74
     entity_density           0.09         0.00          0.00        0.00             0.00
discourse_coherence           0.09         0.06         -0.49       -0.49             0.00
                ttr           0.18        -0.06         -0.99       -0.99             0.50


## Analysis 2 — Aggregated marginal contribution across all subsets

For each feature: the mean metric over every configuration that contains it, minus the mean over
every configuration that lacks it (sizes 2–5 all contribute contexts). A feature whose margin is
negative across the board hurts regardless of which other features accompany it — much stronger
evidence than any single deletion.

In [5]:
marg_rows = []
subset_rows = ab[ab['n_active'] > 0]     # exclude the probe floor from contexts
for i, f in enumerate(FEATURES):
    has = subset_rows[subset_rows['active_idx'].str.split(',').apply(lambda L: str(i) in L)]
    not_ = subset_rows[~subset_rows.index.isin(has.index)]
    marg_rows.append({'feature': f,
        'n_with': len(has), 'n_without': len(not_),
        'm_standard': 100*(has['standard_f1'].mean() - not_['standard_f1'].mean()),
        'm_logo_mean': 100*(has['logo_mean'].mean() - not_['logo_mean'].mean()),
        'm_logo_worst': 100*(has['logo_worst'].mean() - not_['logo_worst'].mean()),
        'm_fold_gpt': 100*(has['fold_gpt'].mean() - not_['fold_gpt'].mean()),
        'm_fold_deepseek': 100*(has['fold_deepseek'].mean() - not_['fold_deepseek'].mean())})
marg = pd.DataFrame(marg_rows)
print('aggregated marginal contribution (pp; + = feature helps on average, - = hurts):')
print(marg.round(2).sort_values('m_logo_worst').to_string(index=False))
marg.to_parquet(f'{OUT_DIR}/ph2_nb6c_marginals.parquet', index=False)

aggregated marginal contribution (pp; + = feature helps on average, - = hurts):
            feature  n_with  n_without  m_standard  m_logo_mean  m_logo_worst  m_fold_gpt  m_fold_deepseek
       targeted_ppl      15         11        0.06        -0.01         -0.37       -0.37             0.04
     entity_density      15         11       -0.01        -0.00         -0.28       -0.28             0.12
discourse_coherence      15         11       -0.03        -0.06         -0.21       -0.21            -0.03
         burstiness      15         11       -0.04        -0.26          0.03        0.03            -0.38
                ttr      15         11       -0.04         0.27          1.04        1.04            -0.31


## Analysis 3 — Best subset by the pre-registered criterion, and the verdict

In [6]:
top = ab.sort_values(['logo_worst','standard_f1'], ascending=False).head(8)
print('top configurations by LOGO worst fold:')
print(top[['label','n_active','standard_f1','logo_mean','logo_worst','worst_gen']]
      .assign(standard_f1=lambda d:(100*d.standard_f1).round(1),
              logo_mean=lambda d:(100*d.logo_mean).round(1),
              logo_worst=lambda d:(100*d.logo_worst).round(1))
      .to_string(index=False))

print()
# verdict logic: a feature is 'harmful' if (a) its removal alone improves logo_worst AND standard,
# and (b) its aggregated marginals are negative on logo_worst
harmful = []
for _, r in loo.iterrows():
    f = r['removed']
    mg = marg[marg['feature']==f].iloc[0]
    if r['d_logo_worst'] > 0.15 and r['d_standard_f1'] >= -0.05 and mg['m_logo_worst'] < 0:
        harmful.append((f, r['d_logo_worst'], mg['m_logo_worst']))

if harmful:
    harmful.sort(key=lambda t: -t[1])
    print('VERDICT — feature(s) satisfying both harm criteria:')
    for f, d1, d2 in harmful:
        print(f'  {f}: removal alone improves LOGO-worst by {d1:+.2f} pp; '
              f'aggregated margin {d2:+.2f} pp')
    print('\nrecommendation: drop the top-listed feature, re-lock Vstat, and record the change')
else:
    print('VERDICT — no feature satisfies both harm criteria; keep all five.')
    print('(differences within ~0.15 pp are treated as noise even for a deterministic fit,')
    print(' because a single test article is worth ~0.09 pp on this split)')

best = ab.sort_values(['logo_worst','standard_f1'], ascending=False).iloc[0]
print(f"\nbest overall subset: {best['label']}  "
      f"(std {100*best['standard_f1']:.1f}, worst {100*best['logo_worst']:.1f})")
print(f"all-5 baseline:      std {100*base['standard_f1']:.1f}, worst {100*base['logo_worst']:.1f}")

top configurations by LOGO worst fold:
                                                       label  n_active  standard_f1  logo_mean  logo_worst worst_gen
without: targeted_ppl + entity_density + discourse_coherence         2         99.7       98.2        95.2       gpt
         without: targeted_ppl + burstiness + entity_density         2         99.6       98.6        95.2       gpt
    without: targeted_ppl + burstiness + discourse_coherence         2         99.7       98.5        94.7       gpt
                 without: targeted_ppl + discourse_coherence         3         99.6       98.2        94.7       gpt
                                       without: targeted_ppl         4         99.6       98.1        94.3       gpt
                                         without: burstiness         4         99.7       98.5        94.2       gpt
                                     without: entity_density         4         99.7       98.0        94.2       gpt
               without: e

## Optional MLP arm (set RUN_MLP_ARM = True; GPU recommended)

Repeats the 27 configurations with the fixed MLP head from NB6b (256, dropout 0.0, LayerNorm,
mini-batch 256, early stopping on val macro-F1). Adds training stochasticity, so read it as a
robustness check on the linear verdict, not as the primary evidence.

In [7]:
if RUN_MLP_ARM:
    import torch, torch.nn as nn
    from sklearn.metrics import f1_score as _f1
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('MLP arm on', DEVICE)

    class Head(nn.Module):
        def __init__(self, d, hidden=256):
            super().__init__()
            self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 2))
        def forward(self, x): return self.net(x)

    def train_mlp(Xtr, ytr, Xva, yva, batch=256, max_epochs=200, patience=20):
        torch.manual_seed(SEED)
        g = torch.Generator().manual_seed(SEED)
        model = Head(Xtr.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        n0, n1 = int((ytr==0).sum()), int((ytr==1).sum())
        w = torch.tensor([(n0+n1)/(2*n0), (n0+n1)/(2*n1)], dtype=torch.float, device=DEVICE)
        lossf = nn.CrossEntropyLoss(weight=w)
        Xtr_t = torch.tensor(Xtr, device=DEVICE); ytr_t = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
        Xva_t = torch.tensor(Xva, device=DEVICE)
        best, best_state, wait = -1, None, 0
        for ep in range(max_epochs):
            model.train()
            perm = torch.randperm(len(Xtr_t), generator=g).to(DEVICE)
            for b in range(0, len(Xtr_t), batch):
                idx = perm[b:b+batch]
                opt.zero_grad(); lossf(model(Xtr_t[idx]), ytr_t[idx]).backward(); opt.step()
            model.eval()
            with torch.no_grad():
                f1 = _f1(yva, model(Xva_t).argmax(1).cpu().numpy(), average='macro')
            if f1 > best + 1e-6:
                best, wait = f1, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= patience: break
        model.load_state_dict(best_state)
        return model

    def mlp_f1(model, X, yy):
        model.eval()
        with torch.no_grad():
            pred = model(torch.tensor(X, device=DEVICE)).argmax(1).cpu().numpy()
        return _f1(yy, pred, average='macro')

    mlp_rows, t0 = [], time.time()
    for k, active in enumerate(configs):
        X = build_X(active)
        m = train_mlp(X[tr_m], y[tr_m], X[va_m], y[va_m])
        std = mlp_f1(m, X[te_m], y[te_m])
        folds = {}
        for gname in gen_list:
            trm = (splits=='train') & ((y==0) | (gens!=gname))
            vam = (splits=='val')   & ((y==0) | (gens!=gname))
            tem = (splits=='test')  & ((y==0) | (gens==gname))
            mm = train_mlp(X[trm], y[trm], X[vam], y[vam])
            folds[gname] = mlp_f1(mm, X[tem], y[tem])
        fv = np.array(list(folds.values()))
        mlp_rows.append({'label': label_of(active), 'n_active': len(active),
                         'standard_f1': std, 'logo_mean': fv.mean(),
                         'logo_worst': fv.min(), 'worst_gen': gen_list[int(fv.argmin())],
                         **{f'fold_{gg}': folds[gg] for gg in gen_list}})
        print(f'[MLP {k+1:2d}/27] {label_of(active):<55} std {100*std:5.1f}  '
              f'worst {100*fv.min():5.1f}', flush=True)
    mlp_ab = pd.DataFrame(mlp_rows)
    mlp_ab.to_parquet(f'{OUT_DIR}/ph2_nb6c_ablation_mlp.parquet', index=False)
    print(f'MLP arm total {time.time()-t0:.0f}s | saved ph2_nb6c_ablation_mlp.parquet')
else:
    print('MLP arm skipped (RUN_MLP_ARM = False)')

MLP arm on cuda
[MLP  1/27] ALL 5                                                   std  99.7  worst  91.8
[MLP  2/27] without: targeted_ppl                                   std  99.6  worst  92.8
[MLP  3/27] without: burstiness                                     std  99.6  worst  92.3
[MLP  4/27] without: ttr                                            std  99.7  worst  89.1
[MLP  5/27] without: entity_density                                 std  99.6  worst  93.3
[MLP  6/27] without: discourse_coherence                            std  99.7  worst  91.7
[MLP  7/27] without: targeted_ppl + burstiness                      std  99.7  worst  92.3
[MLP  8/27] without: targeted_ppl + ttr                             std  99.8  worst  92.3
[MLP  9/27] without: targeted_ppl + entity_density                  std  99.6  worst  93.3
[MLP 10/27] without: targeted_ppl + discourse_coherence             std  99.7  worst  93.2
[MLP 11/27] without: burstiness + ttr                               std  9

## Notes

- **Why the linear arm is the attribution-grade evidence:** it is fully converged and deterministic,
  so a 0.3-pp difference between subsets is a property of the features, not of a random seed. The
  MLP arm re-trains stochastically per subset and is read only as a consistency check.
- **Noise floor:** one test article ≈ 0.09 pp of macro-F1 on this split; LOGO folds with 63–137 AI
  test articles move in steps of ~0.4–0.8 pp per article. Deltas inside these step sizes are not
  meaningful, which is why the verdict thresholds are conservative.
- **If a feature is convicted:** drop it from Vstat, update the feature-order contract everywhere
  (the skills, the plan, Track B, and the thesis's R^5 → R^4 / R^772 dimensions), and record the
  ablation table as the justification. If all five are acquitted, the table still belongs in the
  thesis as the feature-level ablation the examiners will ask for.
- **DeepSeek watch:** the per-fold columns let you check whether the subset that fixes DeepSeek's
  regression is the same one that preserves the GPT gain — the interesting tension this study can
  reveal.